# Document Chunking Strategies
Companion notebook for the **Document Chunking Strategies** section of Chapter 10.

Chunking decides what your retriever can actually find. Chunk size is a tradeoff: smaller chunks improve precision, larger chunks keep more context but add noise.

In [1]:
SAMPLE_POLICY_TEXT = (
    "Refunds are allowed within 30 days of purchase for annual plans. "
    "To request a refund, contact support with your order ID. "
    "Refunds are processed within 5 to 7 business days. "
    "Enterprise plans follow a separate 45 day refund window, except when a custom "
    "contract states otherwise. "
    "Partial refunds are not offered for partially used subscription periods. "
    "Chargebacks initiated outside of this process may result in account suspension."
)

print(SAMPLE_POLICY_TEXT)

Refunds are allowed within 30 days of purchase for annual plans. To request a refund, contact support with your order ID. Refunds are processed within 5 to 7 business days. Enterprise plans follow a separate 45 day refund window, except when a custom contract states otherwise. Partial refunds are not offered for partially used subscription periods. Chargebacks initiated outside of this process may result in account suspension.


## Fixed-size chunking
The book's baseline numbers -- split by characters/tokens with a fixed overlap:

In [2]:
chunk_size = 512
chunk_overlap = 64


def fixed_size_chunks(text: str, size: int, overlap: int) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + size)
        chunks.append(text[start:end])
        start += size - overlap
    return chunks


# A small size here (120/20) so the tradeoff is visible on a short sample text
fixed_chunks = fixed_size_chunks(SAMPLE_POLICY_TEXT, size=120, overlap=20)
for i, c in enumerate(fixed_chunks):
    print(f"[{i}] {c!r}")

[0] 'Refunds are allowed within 30 days of purchase for annual plans. To request a refund, contact support with your order ID'
[1] 't with your order ID. Refunds are processed within 5 to 7 business days. Enterprise plans follow a separate 45 day refun'
[2] 'eparate 45 day refund window, except when a custom contract states otherwise. Partial refunds are not offered for partia'
[3] 't offered for partially used subscription periods. Chargebacks initiated outside of this process may result in account s'
[4] ' result in account suspension.'


Notice a fixed chunk can cut mid-sentence -- exactly the failure mode the book warns about ("a cut between \"allowed\" and \"except when\" can change the meaning").

## Sentence-aware (semantic) chunking
Split at sentence boundaries instead of an exact character count, so meaning stays intact:

In [3]:
import re


def sentence_aware_chunks(text: str, target_size: int) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current = [], ""
    for sentence in sentences:
        candidate = f"{current} {sentence}".strip()
        if current and len(candidate) > target_size:
            chunks.append(current)
            current = sentence
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks


semantic_chunks = sentence_aware_chunks(SAMPLE_POLICY_TEXT, target_size=120)
for i, c in enumerate(semantic_chunks):
    print(f"[{i}] {c!r}")

assert all(c.rstrip().endswith((".", "!", "?")) for c in semantic_chunks), "a chunk ends mid-sentence"
print("\nevery chunk ends on a real sentence boundary")

[0] 'Refunds are allowed within 30 days of purchase for annual plans.'
[1] 'To request a refund, contact support with your order ID. Refunds are processed within 5 to 7 business days.'
[2] 'Enterprise plans follow a separate 45 day refund window, except when a custom contract states otherwise.'
[3] 'Partial refunds are not offered for partially used subscription periods.'
[4] 'Chargebacks initiated outside of this process may result in account suspension.'

every chunk ends on a real sentence boundary


## Parent-child chunking
Embed small **child** chunks for precise search, but return the larger **parent** section as answer-time context -- good for manuals, contracts, and technical docs where the model needs the surrounding section, not just the matched sentence.

In [4]:
def parent_child_chunks(text: str, parent_size: int, child_size: int) -> list[dict]:
    # Parents are grouped on sentence boundaries too -- a raw fixed-size cut here would
    # slice a parent mid-word, which then poisons every child chunked out of it.
    parents = sentence_aware_chunks(text, target_size=parent_size)
    records = []
    for p_idx, parent in enumerate(parents):
        children = sentence_aware_chunks(parent, target_size=child_size)
        for c_idx, child in enumerate(children):
            records.append({
                "parent_id": f"parent-{p_idx}",
                "parent_text": parent,
                "child_id": f"parent-{p_idx}:child-{c_idx}",
                "child_text": child,
            })
    return records


records = parent_child_chunks(SAMPLE_POLICY_TEXT, parent_size=220, child_size=80)
for r in records:
    print(f"{r['child_id']}: child={r['child_text']!r}")
    print(f"  -> retrieving this child returns parent {r['parent_id']}: {r['parent_text'][:60]!r}...")

parent-0:child-0: child='Refunds are allowed within 30 days of purchase for annual plans.'
  -> retrieving this child returns parent parent-0: 'Refunds are allowed within 30 days of purchase for annual pl'...
parent-0:child-1: child='To request a refund, contact support with your order ID.'
  -> retrieving this child returns parent parent-0: 'Refunds are allowed within 30 days of purchase for annual pl'...
parent-0:child-2: child='Refunds are processed within 5 to 7 business days.'
  -> retrieving this child returns parent parent-0: 'Refunds are allowed within 30 days of purchase for annual pl'...
parent-1:child-0: child='Enterprise plans follow a separate 45 day refund window, except when a custom contract states otherwise.'
  -> retrieving this child returns parent parent-1: 'Enterprise plans follow a separate 45 day refund window, exc'...
parent-1:child-1: child='Partial refunds are not offered for partially used subscription periods.'
  -> retrieving this child returns parent paren

Which one to use isn't a universal choice: start with a fixed baseline, add overlap where boundary loss hurts, and move to semantic or parent-child chunking once structure (policy text, contracts, manuals) starts to matter more than raw simplicity.